# TradeSense · Component 03 · Stage E · Notebook E1
### Build the synthetic order-book manipulation sets (validation month only) for testing what the TGNN is for

**Why.** The three finished models read 1-minute trade bars. Order-book manipulation that lives for a few seconds (spoofing, layering, quote stuffing, liquidity pulling) is invisible at that scale, so it is the TGNN's remit. There are no public real labels for such behaviour on Binance Spot, so this notebook creates **labelled synthetic** events inside the real validation day (July) and keeps everything else untouched. Every result built on these sets is reported as *synthetic injection*.

**What it injects (4 patterns × 3 magnitudes, random sides and ranks).**
| pattern | what changes |
|---|---|
| spoof | one large quantity a few ranks from the touch for 1 to 10 s, then cancelled |
| layering | 3 to 6 consecutive ranks get large quantities, staggered, removed together |
| quote stuffing | quantity on the 10 best ranks switches on and off every 100 ms for 3 to 10 s |
| liquidity vacuum | the 3 to 8 best ranks on one side lose 60 to 95% of their quantity for 1 to 5 s |

Prices, spread and mid never change; only the quantity channels (`log_qty`, `d_log_qty`, `cum_log_depth`, `imbalance_top5`) are recomputed with the Stage C formulas and the frozen training scaler. Events never overlap a flagged interval and are at least 60 s apart.

**What it does NOT do.** It opens no August, September or October day and injects nothing into training days. The test-day version is built later, inside the single test opening of D3. It writes only into the new folder `TGNN_StageE_v1`. **No GPU is needed**: use a CPU runtime (about 10 to 20 minutes).

## 1 · Settings (nothing to edit)

In [ ]:
import os, gc, sys, json, time, shutil, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), f'Drive folder not found: {BASE}'
STAGEC_NAME = 'TGNN_StageC_GraphFeatures_v2'; OUT_NAME = 'TGNN_StageE_v1'
N_PER_PAIR = 15          # events per (pattern, magnitude) pair per day and symbol: 4 x 3 x 15 = 180 events
MARGIN = 600; GAP = 600  # clean intervals before an event; minimum distance between events (100 ms intervals)
SEED = 4242
print('Stage C:', STAGEC_NAME, '| output:', OUT_NAME, '| events per set:', 12 * N_PER_PAIR)

Mounted at /content/drive
Stage C: TGNN_StageC_GraphFeatures_v2 | output: TGNN_StageE_v1 | events per set: 180


## 2 · The injector library (written to the Colab working folder, not to Drive)

In [ ]:
%%writefile stage_e_lib.py
"""TradeSense C03 - Stage E library: order-book manipulation INJECTOR (synthetic, labelled as synthetic).
Works on the frozen Stage C tensors (scaled float16). Levels are decoded to quantities, a pattern is added to the quantities of existing price levels
(prices, spread and mid never change), and ONLY the quantity-derived channels are recomputed with the same formulas as Stage C:
  node ch2 log_qty, ch3 d_log_qty, ch4 cum_log_depth, glob ch1 imbalance_top5   (ch1/ch5 distances and glob ch0/ch2 do not depend on quantities).
Intervals outside [start, end+1] of an event are copied untouched. Injection never targets a flagged interval. Training days are never injected."""
import json, os
from pathlib import Path
import numpy as np

K = 25; NN = 50; NCH = 6; GCH = 3; LOGQ, DLQ, CUM = 2, 3, 4; IMB = 1
SCALED_NODE = [1, 2, 3, 4, 5]
PATTERNS = {1: 'spoof', 2: 'layering', 3: 'quote_stuffing', 4: 'liquidity_vacuum'}
MAGS = {1: 'subtle', 2: 'medium', 3: 'easy'}
# magnitude parameters per pattern and level (1 subtle, 2 medium, 3 easy). multipliers are in units of the day-median quantity of that rank
PAR = {1: {1: 3.0, 2: 8.0, 3: 20.0}, 2: {1: 2.0, 2: 5.0, 3: 10.0}, 3: {1: 0.8, 2: 2.0, 3: 4.0}, 4: {1: 0.60, 2: 0.85, 3: 0.95}}
DUR = {1: (10, 100), 2: (20, 150), 3: (30, 100), 4: (10, 50)}          # in 100 ms intervals

# ---------------------------------------------------------------- decode / recompute
def _mu_sd(sc, ch): j = SCALED_NODE.index(ch); return float(sc['node_mean'][j]), float(sc['node_std'][j])
def decode_q(nodes, sc, a, b):
    """quantities of intervals [a,b) from the scaled log_qty channel (float64, >= 0)"""
    m, s = _mu_sd(sc, LOGQ); lq = nodes[a:b, :, LOGQ].astype(np.float64) * s + m; return np.maximum(np.expm1(lq), 0.0)
def encode_channels(q, q_prev, sc, glob_mean, glob_std):
    """scaled (log_qty, d_log_qty, cum_log_depth) of quantities q (m,50) given the quantities of the interval just before (50,) and the scaled imbalance (m,)"""
    lq = np.log1p(q); lq_prev = np.log1p(q_prev)[None]; dlq = lq - np.concatenate([lq_prev, lq[:-1]], 0)
    cum = np.concatenate([np.log1p(np.cumsum(q[:, :K], 1)), np.log1p(np.cumsum(q[:, K:], 1))], 1)
    s5b = q[:, :5].sum(1); s5a = q[:, K:K + 5].sum(1); imb = (s5b - s5a) / np.maximum(s5b + s5a, 1e-12)
    out = []
    for ch, arr in ((LOGQ, lq), (DLQ, dlq), (CUM, cum)): m, s = _mu_sd(sc, ch); out.append((arr - m) / s)
    return out[0], out[1], out[2], (imb - glob_mean) / glob_std

def day_ref(nodes, sc, flags, step=50):
    """per-rank median quantity of the day (sampled every `step` intervals, flag-free intervals only): the unit of injection magnitude"""
    idx = np.flatnonzero(flags[::step] == 0) * step
    m, s = _mu_sd(sc, LOGQ); lq = nodes[idx, :, LOGQ].astype(np.float64) * s + m; return np.median(np.maximum(np.expm1(lq), 0.0), 0)

# ---------------------------------------------------------------- pattern definitions: additive/multiplicative change of q over the event
def pattern_delta(q_span, pat, mag, side, rng, ref, dur):
    """returns the modified quantities q' for the intervals of the event (dur, 50)"""
    q = q_span.copy(); off = 0 if side == 0 else K; par = PAR[pat][mag]; meta = {}
    if pat == 1:                                                   # spoof: one large order a few ranks away from the touch, then cancelled
        r = int(rng.integers(1, 6)); q[:, off + r] += par * ref[off + r]; meta = {'ranks': [r]}
    elif pat == 2:                                                 # layering: 3-6 consecutive ranks, staggered placement, removed together
        L = int(rng.integers(3, 7)); r0 = int(rng.integers(1, 5)); ranks = list(range(r0, r0 + L)); meta = {'ranks': ranks}
        for i, r in enumerate(ranks):
            st = min(i * int(rng.integers(0, 5)), max(dur - 2, 0)); q[st:, off + r] += par * ref[off + r] * rng.uniform(0.7, 1.3)
    elif pat == 3:                                                 # quote stuffing: rapid on/off of quantity on the 10 best ranks
        on = (np.arange(dur) % 2 == 0); q[on, off:off + 10] *= (1.0 + par); meta = {'ranks': list(range(10))}
    elif pat == 4:                                                 # liquidity vacuum: the k best ranks lose most of their quantity
        k = int(rng.integers(3, 9)); q[:, off:off + k] *= (1.0 - par); meta = {'ranks': list(range(k))}
    return q, meta

# ---------------------------------------------------------------- placement
def place_events(flags, win_end, n_events, rng, margin=600, gap=600, durations=DUR, patterns=(1, 2, 3, 4), mags=(1, 2, 3)):
    """random event slots inside long flag-free stretches: [start, end] (inclusive), margin intervals of clean context before and 50 after, at least `gap` intervals between events.
    n_events = events per (pattern, magnitude) pair."""
    n = len(flags); bad = np.concatenate([[0], np.cumsum((flags != 0).astype(np.int64))]); taken = np.zeros(n, bool); ev = []
    jobs = [(p, m) for p in patterns for m in mags for _ in range(n_events)]; rng.shuffle(jobs)
    for p, m in jobs:
        dur = int(rng.integers(durations[p][0], durations[p][1] + 1))
        for _try in range(200):
            s = int(rng.integers(margin, n - dur - 60)); e = s + dur - 1
            if bad[e + 52] - bad[s - margin] != 0: continue                # whole stretch must be flag-free
            if taken[max(s - gap, 0):e + gap].any(): continue
            taken[s:e + 1] = True; ev.append({'start': s, 'end': e, 'pattern': p, 'mag': m, 'side': int(rng.integers(0, 2))}); break
    ev.sort(key=lambda d: d['start']); return ev

# ---------------------------------------------------------------- apply
def inject_day(nodes, glob, flags, sc, events, seed):
    """returns copies (nodes2, glob2) and an interval label array; events must come from place_events (flag-free, separated)"""
    nodes2 = nodes.copy(); glob2 = glob.copy(); n = len(flags); pat_arr = np.zeros(n, np.int8); mag_arr = np.zeros(n, np.int8); rng = np.random.default_rng(seed)
    ref = day_ref(nodes, sc, flags); gm = float(sc['glob_mean'][IMB]); gs = float(sc['glob_std'][IMB]); out = []
    for e in events:
        s, t, p, m, side = e['start'], e['end'], e['pattern'], e['mag'], e['side']; dur = t - s + 1
        q = decode_q(nodes, sc, s - 1, t + 2)                          # rows: s-1 (context), s..t (event), t+1 (cancel interval)
        q_ev, meta = pattern_delta(q[1:1 + dur], p, m, side, rng, ref, dur)
        qn = q.copy(); qn[1:1 + dur] = q_ev                              # interval t+1 keeps the original (cancelled) quantities
        lq, dl, cu, im = encode_channels(qn[1:], q[0], sc, gm, gs)
        nodes2[s:t + 2, :, LOGQ] = lq; nodes2[s:t + 2, :, DLQ] = dl; nodes2[s:t + 2, :, CUM] = cu; glob2[s:t + 2, IMB] = im
        pat_arr[s:t + 1] = p; mag_arr[s:t + 1] = m; out.append({**e, **meta, 'pattern_name': PATTERNS[p], 'mag_name': MAGS[m]})
    return nodes2, glob2, pat_arr, mag_arr, out

def build_injected_set(src_dir, dst_dir, sc, n_per_pair, seed, margin=600, gap=600):
    """one Stage C set -> injected set folder with the same layout plus event_pattern.npy, event_mag.npy, events.json. Training sets are refused."""
    src = Path(src_dir); dst = Path(dst_dir); sm = json.loads((src / 'set_summary.json').read_text())
    if sm['split'] == 'train': raise PermissionError('training sets are never injected')
    nodes = np.load(src / 'nodes.npy', mmap_mode='r'); glob = np.load(src / 'glob.npy'); flags = np.load(src / 'flags.npy'); win = np.load(src / 'win_end.npy'); rng = np.random.default_rng(seed)
    ev = place_events(flags, win, n_per_pair, rng, margin=margin, gap=gap); n2, g2, pa, ma, evl = inject_day(np.asarray(nodes), glob, flags, sc, ev, seed + 1)
    dst.mkdir(parents=True, exist_ok=True); np.save(dst / 'nodes.npy', n2.astype(np.float16)); np.save(dst / 'glob.npy', g2.astype(np.float16)); np.save(dst / 'flags.npy', flags); np.save(dst / 'win_end.npy', win)
    np.save(dst / 'event_pattern.npy', pa); np.save(dst / 'event_mag.npy', ma); (dst / 'events.json').write_text(json.dumps(evl, indent=1))
    sm2 = dict(sm); sm2.update({'injected': True, 'synthetic_events': len(evl), 'source_set': str(src.name), 'injection_seed': seed}); (dst / 'set_summary.json').write_text(json.dumps(sm2, indent=1)); return evl

# ---------------------------------------------------------------- tests (synthetic data, no real data involved)
def make_toy(n=6000, seed=0):
    """a tiny consistent Stage-C-like day built with the SAME formulas: random quantities -> scaled channels (so decode/encode must be exact up to float16)"""
    rng = np.random.default_rng(seed); q = np.exp(rng.normal(3.0, 0.8, size=(n, NN))); q = np.where(rng.random((n, NN)) < 0.08, q * rng.uniform(1.5, 4, size=(n, NN)), q)
    for i in range(1, n):
        keep = rng.random(NN) < 0.8; q[i] = np.where(keep, q[i - 1], q[i])                                    # sparse changes
    sc = {'node_mean': [0.0, 3.0, 0.0, 6.0, 0.0], 'node_std': [1.0, 1.3, 0.8, 1.5, 1.0], 'glob_mean': [0.0, 0.0, 0.0], 'glob_std': [1.0, 0.4, 1.0]}
    nodes = np.zeros((n, NN, NCH), np.float32); glob = np.zeros((n, GCH), np.float32); lq, dl, cu, im = encode_channels(q, q[0], sc, 0.0, 0.4)
    nodes[:, :, 0] = np.where(np.arange(NN) < K, -1, 1); nodes[:, :, LOGQ] = lq; nodes[:, :, DLQ] = dl; nodes[:, :, CUM] = cu; nodes[:, :, 1] = rng.normal(size=(n, NN)); nodes[:, :, 5] = rng.normal(size=(n, NN)); glob[:, IMB] = im
    flags = np.zeros(n, np.uint8); flags[:700] = 16; flags[6000:6040] = 8
    return nodes.astype(np.float16), glob.astype(np.float16), flags, sc, q
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    nodes, glob, flags, sc, q_true = make_toy(n=12000); n = len(flags); rng = np.random.default_rng(1)
    # H01 decode gives back the quantities (float16 tolerance)
    qd = decode_q(nodes, sc, 0, n); rel = np.abs(qd - q_true) / np.maximum(q_true, 1.0); add('H01', 'decoded quantities equal the true quantities (float16 tolerance)', float(np.median(rel)) < 2e-3 and float(rel.max()) < 3e-2, f'median {np.median(rel):.1e} max {rel.max():.1e}')
    # H02 re-encoding unchanged quantities reproduces the stored channels
    lq, dl, cu, im = encode_channels(qd[1000:1100], qd[999], sc, 0.0, 0.4); add('H02', 'recomputed log_qty, d_log_qty, cum_log_depth and imbalance of untouched data equal the stored channels', np.abs(lq - nodes[1000:1100, :, LOGQ].astype(np.float64)).max() < 0.02 and np.abs(dl - nodes[1000:1100, :, DLQ].astype(np.float64)).max() < 0.05 and np.abs(cu - nodes[1000:1100, :, CUM].astype(np.float64)).max() < 0.05 and np.abs(im - glob[1000:1100, IMB].astype(np.float64)).max() < 0.05)
    # H03 placement rules
    ev = place_events(flags, None, 3, rng, margin=100, gap=100); ok = len(ev) == 36 and all(e['start'] >= 100 for e in ev); bad = np.concatenate([[0], np.cumsum(flags != 0)])
    ok2 = all(bad[e['end'] + 52] - bad[e['start'] - 100] == 0 for e in ev); ok3 = all(ev[i + 1]['start'] - ev[i]['end'] >= 100 for i in range(len(ev) - 1)); add('H03', '36 events placed (3 per pattern/magnitude), never over flagged intervals, with clean context and gaps', ok and ok2 and ok3, f'{len(ev)} events')
    # H04 untouched outside the event: intervals before start and after end+1 are identical, all pre-existing channels except the four recomputed ones unchanged
    n2, g2, pa, ma, evl = inject_day(nodes, glob, flags, sc, ev, 5); idx = np.arange(n); inside = np.zeros(n, bool)
    for e in ev: inside[e['start']:e['end'] + 2] = True
    same = np.array_equal(n2[~inside], nodes[~inside]) and np.array_equal(g2[~inside], glob[~inside]); other = np.array_equal(n2[:, :, [0, 1, 5]], nodes[:, :, [0, 1, 5]]) and np.array_equal(g2[:, [0, 2]], glob[:, [0, 2]])
    add('H04', 'intervals outside every event are byte-identical; distance, side and spread/mid channels are never changed', same and other)
    # H05 causality: the first changed interval is the event start; nothing before it changes
    first = [int(np.flatnonzero((n2[e['start'] - 100:e['end'] + 2] != nodes[e['start'] - 100:e['end'] + 2]).any((1, 2)))[0]) - 100 for e in ev[:12]]; add('H05', 'the first changed interval of every event is its start (nothing earlier is touched)', all(f == 0 for f in first), first[:6])
    # H06 spoof by hand: extra quantity on the chosen rank equals magnitude x day reference, restored after the event
    ref = day_ref(nodes, sc, flags); sp = [e for e in evl if e['pattern'] == 1][0]; s0, e0 = sp['start'], sp['end']; r = sp['ranks'][0] + (0 if sp['side'] == 0 else K)
    qa = decode_q(n2, sc, s0 - 1, e0 + 3); qb = decode_q(nodes, sc, s0 - 1, e0 + 3); add_q = (qa[1:1 + (e0 - s0 + 1), r] - qb[1:1 + (e0 - s0 + 1), r]); target = PAR[1][sp['mag']] * ref[r]
    add('H06', 'spoof: the added quantity equals the magnitude x day reference of that rank during the whole event and is gone after it', abs(np.median(add_q) - target) / target < 0.03 and abs(qa[-1, r] - qb[-1, r]) / max(qb[-1, r], 1) < 0.03, f'median {np.median(add_q):.1f} target {target:.1f}')
    # H07 vacuum removes quantity, stuffing alternates, layering adds on 3+ ranks
    vac = [e for e in evl if e['pattern'] == 4][0]; qv = decode_q(n2, sc, vac['start'], vac['end'] + 1); qo = decode_q(nodes, sc, vac['start'], vac['end'] + 1); off = 0 if vac['side'] == 0 else K; k = len(vac['ranks'])
    ratio = qv[:, off:off + k].sum() / max(qo[:, off:off + k].sum(), 1e-9); stf = [e for e in evl if e['pattern'] == 3][0]; qs = decode_q(n2, sc, stf['start'], stf['start'] + 6); qso = decode_q(nodes, sc, stf['start'], stf['start'] + 6); o2 = 0 if stf['side'] == 0 else K
    alt = (qs[:, o2] / np.maximum(qso[:, o2], 1e-9)); lay = [e for e in evl if e['pattern'] == 2][0]
    add('H07', 'vacuum keeps the expected share of quantity, stuffing alternates on/off, layering touches 3 to 6 ranks', abs(ratio - (1 - PAR[4][vac['mag']])) < 0.06 and alt[0] > 1.5 and abs(alt[1] - 1) < 0.05 and 3 <= len(lay['ranks']) <= 6, f'vacuum kept {ratio:.2f}; stuffing {alt[:3].round(2).tolist()}')
    # H08 labels, determinism, training refusal
    lab_ok = all((pa[e['start']:e['end'] + 1] == e['pattern']).all() and (ma[e['start']:e['end'] + 1] == e['mag']).all() for e in evl) and int((pa > 0).sum()) == sum(e['end'] - e['start'] + 1 for e in evl)
    n3, g3, *_ = inject_day(nodes, glob, flags, sc, ev, 5); det = np.array_equal(n2, n3) and np.array_equal(g2, g3)
    d = tmp / 'src_train'; d.mkdir(parents=True, exist_ok=True); (d / 'set_summary.json').write_text(json.dumps({'split': 'train'})); refused = False
    try: build_injected_set(d, tmp / 'dst', sc, 1, 0)
    except PermissionError: refused = True
    add('H08', 'interval labels equal the events, the same seed gives identical tensors, training sets are refused', lab_ok and det and refused)
    # H09 end-to-end on a folder: val-like set written, readable by the D1 loader layout, injected count as expected
    sd = tmp / 'src_val'; sd.mkdir(parents=True, exist_ok=True); np.save(sd / 'nodes.npy', nodes); np.save(sd / 'glob.npy', glob); np.save(sd / 'flags.npy', flags); np.save(sd / 'win_end.npy', np.arange(700, n - 2, 5).astype(np.int32)); (sd / 'set_summary.json').write_text(json.dumps({'split': 'val', 'n_intervals': n}))
    evx = build_injected_set(sd, tmp / 'dst_val', sc, 2, 3, margin=100, gap=100); fl = sorted(p.name for p in (tmp / 'dst_val').iterdir()); add('H09', 'a validation set is written with labels and events; summary says injected', len(evx) == 24 and 'event_pattern.npy' in fl and json.loads((tmp / 'dst_val' / 'set_summary.json').read_text())['injected'] is True, fl)
    return out

Writing stage_e_lib.py


## 3 · Safety checks: Stage C, frozen scaler, validation days only, frozen configuration

In [ ]:
sys.path.insert(0, str(Path.cwd())); import stage_e_lib as E
LIB_SHA = E.__file__ and __import__('hashlib').sha256(Path('stage_e_lib.py').read_bytes()).hexdigest()
SC_DIR = BASE / STAGEC_NAME; assert (SC_DIR / 'sets').is_dir(), f'Stage C folder not found: {SC_DIR}'
sc_file = json.loads((SC_DIR / 'config' / 'scaler_train_only.json').read_text()); SC = sc_file['scaler']; SC_SHA = sc_file['sha256']
SYMS = ['BTCUSDT', 'ETHUSDT']; VAL_DAYS = ['2024-07-01']; KEYS = [f'{s}_{d}' for d in VAL_DAYS for s in SYMS]
for k in KEYS:
    sm = json.loads((SC_DIR / 'sets' / k / 'set_summary.json').read_text()); assert sm['split'] == 'val', f'{k} is not a validation set'
    assert sm.get('scaler_sha256') in (None, SC_SHA) or str(sm['scaler_sha256'])[:16] == SC_SHA[:16], f'{k}: scaler differs from the frozen scaler'
OUT = BASE / OUT_NAME; (OUT / 'config').mkdir(parents=True, exist_ok=True); (OUT / 'reports').mkdir(exist_ok=True); (OUT / 'val_injected' / 'sets').mkdir(parents=True, exist_ok=True)
FROZEN = {'stagec': STAGEC_NAME, 'scaler_sha256': SC_SHA, 'n_per_pair': N_PER_PAIR, 'margin': MARGIN, 'gap': GAP, 'seed': SEED, 'params': {str(k): v for k, v in E.PAR.items()}, 'durations': {str(k): v for k, v in E.DUR.items()}, 'library_sha256': LIB_SHA, 'val_days': VAL_DAYS}
def _canon(o): return __import__('hashlib').sha256(json.dumps(o, sort_keys=True).encode()).hexdigest()
fp = OUT / 'config' / 'frozen_config_e1.json'
if not fp.exists(): fp.write_text(json.dumps({'config': FROZEN, 'sha256': _canon(FROZEN)}, indent=1)); print('frozen config written')
else: assert json.loads(fp.read_text())['sha256'] == _canon(FROZEN), 'settings or library differ from the saved frozen configuration; do not change them while resuming'; print('frozen config matches')
print('scaler', SC_SHA[:16], '| validation sets:', KEYS, '| library', LIB_SHA[:12])

frozen config written
scaler 2bfe9d4985698ba2 | validation sets: ['BTCUSDT_2024-07-01', 'ETHUSDT_2024-07-01'] | library a67aa38fb95a


## 4 · Tests of the injector on synthetic data (decode/encode exact, nothing outside an event changes, causality, patterns by hand, labels, training sets refused)

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as t_: ER = E.unit_tests(t_)
display(pd.DataFrame(ER, columns=['id', 'test', 'pass', 'detail'])); assert all(r[2] for r in ER), 'A test failed: stop here and send me this table.'
print(f'tests: {sum(r[2] for r in ER)}/{len(ER)} passed')

,id,test,pass,detail
0,H01,decoded quantities equal the true quantities (...,True,median 7.6e-05 max 1.3e-03
1,H02,"recomputed log_qty, d_log_qty, cum_log_depth a...",True,
2,H03,"36 events placed (3 per pattern/magnitude), ne...",True,36 events
3,H04,intervals outside every event are byte-identic...,True,
4,H05,the first changed interval of every event is i...,True,"[0, 0, 0, 0, 0, 0]"
5,H06,spoof: the added quantity equals the magnitude...,True,median 391.7 target 391.8
6,H07,"vacuum keeps the expected share of quantity, s...",True,"vacuum kept 0.40; stuffing [3.0, 1.0, 3.0]"
7,H08,"interval labels equal the events, the same see...",True,
8,H09,a validation set is written with labels and ev...,True,"['event_mag.npy', 'event_pattern.npy', 'events..."


tests: 9/9 passed


## 5 · Build the injected validation sets (resumable; Stage C is only read)

In [ ]:
t0 = time.time(); EV = {}
for i, k in enumerate(KEYS):
    dst = OUT / 'val_injected' / 'sets' / k
    if (dst / 'set_summary.json').exists() and (dst / 'events.json').exists():
        EV[k] = json.loads((dst / 'events.json').read_text()); print(k, 'already built:', len(EV[k]), 'events'); continue
    if dst.exists(): shutil.rmtree(dst)           # an unfinished folder from an interrupted run is rebuilt from scratch (the source is read only)
    EV[k] = E.build_injected_set(SC_DIR / 'sets' / k, dst, SC, N_PER_PAIR, SEED + i, margin=MARGIN, gap=GAP); print(f'{k}: {len(EV[k])} events written ({time.time()-t0:.0f} s)'); gc.collect()

BTCUSDT_2024-07-01: 180 events written (12 s)
ETHUSDT_2024-07-01: 180 events written (22 s)


## 6 · Checks on the real injected sets

In [ ]:
RT = []
def R(i, nm, ok, det=''): RT.append((i, nm, bool(ok), str(det)))
rows = []; ok_same = True; ok_clean = True; mx = {}
for k in KEYS:
    src = SC_DIR / 'sets' / k; dst = OUT / 'val_injected' / 'sets' / k; n0 = np.load(src / 'nodes.npy', mmap_mode='r'); n1 = np.load(dst / 'nodes.npy', mmap_mode='r'); g0 = np.load(src / 'glob.npy'); g1 = np.load(dst / 'glob.npy')
    pa = np.load(dst / 'event_pattern.npy'); ma = np.load(dst / 'event_mag.npy'); fl = np.load(dst / 'flags.npy'); touched = np.zeros(len(pa), bool)
    for e in EV[k]: touched[e['start']:e['end'] + 2] = True
    CH = 200_000; diff_outside = 0
    for a in range(0, len(pa), CH):
        b = min(a + CH, len(pa)); dif = (np.asarray(n0[a:b]) != np.asarray(n1[a:b])).any((1, 2)) | (g0[a:b] != g1[a:b]).any(1); diff_outside += int((dif & ~touched[a:b]).sum())
    ok_same &= diff_outside == 0; ok_clean &= not (fl[touched] != 0).any()
    for e in EV[k]:
        s, t = e['start'], e['end']; off = 0 if e['side'] == 0 else 25; r_ = e['ranks'][0] + off
        d = np.abs(np.asarray(n1[s:t + 1, r_, 2], np.float32) - np.asarray(n0[s:t + 1, r_, 2], np.float32)).max(); mx.setdefault((e['pattern_name'], e['mag_name']), []).append(float(d))
    rows.append({'set': k, 'events': len(EV[k]), 'event intervals': int((pa > 0).sum()), 'share of day %': 100 * float((pa > 0).mean()), 'changed outside events': diff_outside})
display(pd.DataFrame(rows).round(3))
R('S01', 'only intervals inside events differ from Stage C (all other tensor values identical)', ok_same)
R('S02', 'no event touches a flagged interval', ok_clean)
R('S03', 'every pattern and magnitude is present in every set', all(len({(e['pattern'], e['mag']) for e in EV[k]}) == 12 for k in KEYS))
try: E_t = E.S if False else None
except Exception: pass
import importlib; R('S04', 'no test or training day was read or written (validation sets only)', all(k.split('_', 1)[1] in VAL_DAYS for k in KEYS) and not (BASE / 'TGNN_StageD_v1' / 'config' / 'freeze_d3.json').exists())
R('S05', 'values stay finite and inside the float16 range', all(np.isfinite(np.asarray(np.load(OUT / 'val_injected' / 'sets' / k / 'nodes.npy', mmap_mode='r')[::20])).all() for k in KEYS))
display(pd.DataFrame([{'pattern': p, 'magnitude': m, 'events': len(v), 'median max |change| of the injected rank (scaled log_qty)': float(np.median(v))} for (p, m), v in sorted(mx.items())]).round(3))
display(pd.DataFrame(RT, columns=['id', 'test', 'pass', 'detail']))

,set,events,event intervals,share of day %,changed outside events
0,BTCUSDT_2024-07-01,180,11154,1.291,0
1,ETHUSDT_2024-07-01,180,9997,1.157,0


,pattern,magnitude,events,median max |change| of the injected rank (scaled log_qty)
0,layering,easy,30,1.914
1,layering,medium,30,0.958
2,layering,subtle,30,0.729
3,liquidity_vacuum,easy,30,2.991
4,liquidity_vacuum,medium,30,2.227
5,liquidity_vacuum,subtle,30,1.150
6,quote_stuffing,easy,30,2.129
7,quote_stuffing,medium,30,1.427
8,quote_stuffing,subtle,30,0.758
9,spoof,easy,30,2.228


,id,test,pass,detail
0,S01,only intervals inside events differ from Stage...,True,
1,S02,no event touches a flagged interval,True,
2,S03,every pattern and magnitude is present in ever...,True,
3,S04,no test or training day was read or written (v...,True,
4,S05,values stay finite and inside the float16 range,True,


## 7 · Saved report (never overwritten) and the SHARE BLOCK

In [ ]:
ALL = [(r[0], r[1], r[2], r[3]) for r in ER] + RT; ok = all(r[2] for r in ALL); VERDICT = 'E1 COMPLETE' if ok else 'E1 INCOMPLETE (a check failed)'
rep = {'verdict': VERDICT, 'tests': ALL, 'frozen': FROZEN, 'events': {k: len(v) for k, v in EV.items()}, 'by_pattern': {k: pd.Series([e['pattern_name'] + '/' + e['mag_name'] for e in v]).value_counts().to_dict() for k, v in EV.items()}, 'note': 'synthetic injection into real validation data; no test day opened'}
rp = OUT / 'reports' / 'e1_report.json'
if not rp.exists(): rp.write_text(json.dumps(rep, indent=1, default=str))
L = ['==== E1 SHARE BLOCK (copy everything between the two ==== lines) ====', f'{VERDICT} | tests passed {sum(r[2] for r in ALL)}/{len(ALL)} | test days opened: NO | lib {LIB_SHA[:12]} | scaler {SC_SHA[:12]}', f'events per set: ' + ', '.join(f'{k} {len(v)}' for k, v in EV.items()), 'failed tests: ' + (', '.join(r[0] for r in ALL if not r[2]) or 'none'), '==== END ====']
print('\n'.join(L)); (OUT / 'reports' / 'e1_share_block.txt').write_text('\n'.join(L)) if not (OUT / 'reports' / 'e1_share_block.txt').exists() else None

==== E1 SHARE BLOCK (copy everything between the two ==== lines) ====
E1 COMPLETE | tests passed 14/14 | test days opened: NO | lib a67aa38fb95a | scaler 2bfe9d498569
events per set: BTCUSDT_2024-07-01 180, ETHUSDT_2024-07-01 180
failed tests: none
==== END ====


262